In [0]:
-- Analyze:
-- * late deliveries %
-- * avg review score by delay bucket
with raw as (
  SELECT
    CASE
      WHEN order_delivered_customer_date > order_estimated_delivery_date THEN 'Delayed'
      ELSE 'On-Time'
    END AS delivery_status,
    COUNT(DISTINCT order_id) AS orders,
    ROUND(AVG(review_score), 2) AS avg_review_score,
    ROUND(AVG(payment_value), 2) AS avg_order_value
  FROM
    workspace.default.final_quick_comm_dataset
  WHERE
    order_status = 'delivered'
  GROUP BY
    1
)
SELECT
  ROUND(
    SUM(CASE WHEN delivery_status = 'Delayed' THEN orders ELSE 0 END) * 100.00
      / SUM(orders),
    2
  ) AS pct_late_deliveries,
  MAX(CASE WHEN delivery_status = 'Delayed' THEN avg_review_score END) AS avg_review_score_delayed_bucket
FROM
  raw;